In [ ]:
# pip install libraries
!pip install timm

In [ ]:
#Import libraries
import pandas as pd
from glob import glob
import os
import re
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import random_split, DataLoader
import torch.nn as nn
import torchvision.models as models
import timm
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from collections import Counter
import numpy as np
import torch.nn.functional as F
import random
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm

In [ ]:
# import os
# import pandas as pd
# from glob import glob

# # Define the main dataset directory
# dataset_dir = "/content/drive/MyDrive/Thesis/POM-IMG"

# # Function to process all weeks, starting from week-2
# def process_dataset(dataset_dir):
#     week_folders = sorted([f for f in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, f)) and "week-4" in f])

#     if not week_folders:
#         print("No valid weeks found (week-2 and later). Check dataset structure.")
#         return

#     for week in week_folders:
#         tree_data = []  # List to store (image_path, metadata_row)
#         week_path = os.path.join(dataset_dir, week)
#         metadata_columns = []  # Placeholder for column names

#         # Get all date folders inside each week
#         date_folders = sorted([f for f in os.listdir(week_path) if os.path.isdir(os.path.join(week_path, f))])

#         for date in date_folders:
#             date_path = os.path.join(week_path, date)

#             # Get all tree folders inside each date
#             tree_folders = sorted([f for f in os.listdir(date_path) if os.path.isdir(os.path.join(date_path, f))])

#             for tree_folder in tree_folders:
#                 tree_path = os.path.join(date_path, tree_folder)

#                 # Locate semgnt_img folder and CSV file
#                 segment_img_path = os.path.join(tree_path, "semgnt_img")
#                 merged_data_csv = os.path.join(tree_path, "merged_data.csv")

#                 print(f"\n Checking Tree Folder: {tree_folder}")
#                 print(f"Looking for images in: {segment_img_path}")

#                 # Skip if the image folder or CSV file is missing
#                 if not os.path.exists(segment_img_path):
#                     print(f" Skipping {tree_folder} in {date}: ❌ Missing semgnt_img folder")
#                     continue
#                 if not os.path.exists(merged_data_csv):
#                     print(f"Skipping {tree_folder} in {date}: ❌ Missing merged_data.csv")
#                     continue

#                 # Load all images (try different formats)
#                 images = sorted(glob(os.path.join(segment_img_path, "*.JPG")))  # Adjust file type if needed
#                 if not images:
#                     images = sorted(glob(os.path.join(segment_img_path, "*.png")))
#                 if not images:
#                     images = sorted(glob(os.path.join(segment_img_path, "*.jpeg")))

#                 print(f"Found {len(images)} images")
#                 if len(images) > 0:
#                     print("Example image path:", images[0])

#                 if not images:
#                     print(f"Skipping {tree_folder} in {date}: ❌ No images found in semgnt_img")
#                     continue

#                 # Load CSV file
#                 try:
#                     metadata_df = pd.read_csv(merged_data_csv)
#                     metadata_columns = metadata_df.columns.tolist()
#                     print(f"CSV loaded: {len(metadata_df)} rows found")
#                 except Exception as e:
#                     print(f"Error reading CSV for {tree_folder} in {date}: {e}")
#                     continue

#                 num_images = len(images)
#                 num_rows = len(metadata_df)

#                 if num_images == 0 and num_rows == 0:
#                     print(f"Skipping {tree_folder} in {date}: ❌ No images and no metadata found.")
#                     continue
#                 elif num_images == 0:
#                     print(f"Skipping {tree_folder} in {date}: ❌ No images found.")
#                     continue
#                 elif num_rows == 0:
#                     print(f"Skipping {tree_folder} in {date}: ❌ No metadata rows found.")
#                     continue

#                 # **Ensure strict 1:1 pairing between images and metadata**
#                 if num_images == num_rows:
#                     pairs = zip(images, metadata_df.iterrows())  # Direct match
#                 elif num_images > num_rows:
#                     # More images than metadata → Repeat last row
#                     repeated_metadata = metadata_df.to_dict(orient="records") + [metadata_df.iloc[-1].to_dict()] * (num_images - num_rows)
#                     repeated_metadata = [pd.Series(row) for row in repeated_metadata]  # Convert dicts back to Series
#                     pairs = zip(images, repeated_metadata)
#                 else:
#                     # More metadata than images → Repeat last image
#                     repeated_images = images + [images[-1]] * (num_rows - num_images)
#                     pairs = zip(repeated_images, metadata_df.iterrows())

#                 # Store image-metadata pairs
#                 for img_path, metadata_row in pairs:
#                     if isinstance(metadata_row, tuple):  # If metadata is from .iterrows(), extract row
#                         metadata_row = metadata_row[1]
#                     tree_data.append([week, date, tree_folder, img_path, *metadata_row.to_list()])

#         # Convert to DataFrame for easier analysis
#         if tree_data:
#             tree_df = pd.DataFrame(tree_data, columns=["Week", "Date", "Tree_ID", "Image_Path"] + metadata_columns)

#             # Save separate CSV for each week
#             output_csv = os.path.join(dataset_dir, f"processed_{week}.csv")
#             tree_df.to_csv(output_csv, index=False)

#             print(f" Processed data for {week} saved to {output_csv}")
#         else:
#             print(f" No valid data found for {week}, skipping CSV creation.")

# # Run the processing function
# process_dataset(dataset_dir)

In [ ]:
  # def rename_columns(merged_df):
  #     """Forcefully renames specific columns after merging to match the correct format."""
  #     rename_dict = {
  #         "Temp ֲ°C Avg.": "Temp °C Avg.",
  #         "Humidity % Avg.": "Humidity % Avg.",
  #         "Solar Radiation W/mֲ² Avg.": "Solar Radiation W/m² Avg.",
  #         "Wind Speed m/sec Avg.": "Wind Speed m/sec Avg.",
  #         "Wind Dir Avg.": "Wind Dir Avg.",
  #         "TC ֲ°C Avg.": "TC °C Avg.",
  #         "Atmospheric Pressure mb Avg.": "Atmospheric Pressure mb Avg.",
  #         "Dew Point Cֲ° Avg.": "Dew Point C° Avg.",
  #         "TC Cֲ° Min.": "TC C° Min.",
  #         "Solar Radiation W/mֲ² Max.": "Solar Radiation W/m² Max.",
  #         "Solar Radiation W/mֲ² Min.": "Solar Radiation W/m² Min."
  #     }

  #     merged_df = merged_df.rename(columns=rename_dict)
  #     return merged_df

In [ ]:
# # Rename columns AFTER merging
# week_df = pd.read_csv("/content/drive/MyDrive/Thesis/POM-IMG/test_row3.csv") # Load the CSV into a DataFrame
# week = rename_columns(week_df) # Pass the DataFrame to rename_columns
# week.to_csv("/content/drive/MyDrive/Thesis/POM-IMG/test_row3.csv", index=False, encoding='utf-8-sig')

In [ ]:
# # **Manually Assign Train & Test Trees**
# train_trees = ["Y1", "Y1W", "Y1W2", "B", "BW", "BW2", "Y2", "Y2W", "Y2W2"]
# test_trees = ["Y1W3", "BW3", "Y2W3"]  # Explicitly set test trees

# # **Filter the Full Dataset Based on the New Split**
# train_data = full_dataset.data[full_dataset.data["Tree_Prefix"].isin(train_trees)]
# test_data = full_dataset.data[full_dataset.data["Tree_Prefix"].isin(test_trees)]

# # **Save Train & Test CSVs**
# train_csv = "/content/drive/MyDrive/Thesis/POM-IMG/train_data.csv"
# test_csv = "/content/drive/MyDrive/Thesis/POM-IMG/test_data.csv"

# train_data.to_csv(train_csv, index=False)
# test_data.to_csv(test_csv, index=False)

# print(f"Train trees: {len(train_trees)}, Test trees: {len(test_trees)}")

In [ ]:
# #crea spertr CSV for rows
# import pandas as pd

# # Paths to original CSVs
# train_csv_path = "/content/drive/MyDrive/Thesis/POM-IMG/train_data_p.csv"
# test_csv_path = "/content/drive/MyDrive/Thesis/POM-IMG/test_data_p.csv"

# # Load datasets
# train_data = pd.read_csv(train_csv_path)
# test_data = pd.read_csv(test_csv_path)

# # Filter by Row_Num (since it already exists)
# train_row3 = train_data[train_data["Row_Number"] == 3].reset_index(drop=True)
# train_row4 = train_data[train_data["Row_Number"] == 4].reset_index(drop=True)
# test_row3 = test_data[test_data["Row_Number"] == 3].reset_index(drop=True)
# test_row4 = test_data[test_data["Row_Number"] == 4].reset_index(drop=True)

# # Save separate CSVs
# train_row3.to_csv("/content/drive/MyDrive/Thesis/POM-IMG/train_row3.csv", index=False)
# train_row4.to_csv("/content/drive/MyDrive/Thesis/POM-IMG/train_row4.csv", index=False)
# test_row3.to_csv("/content/drive/MyDrive/Thesis/POM-IMG/test_row3.csv", index=False)
# test_row4.to_csv("/content/drive/MyDrive/Thesis/POM-IMG/test_row4.csv", index=False)

# print("CSVs successfully separated and saved!")


In [ ]:
#creating CSV triplet for each image

# # Create folders if they don't exist
# os.makedirs("/content/drive/MyDrive/Thesis/POM-IMG/train_row3", exist_ok=True)
# os.makedirs("/content/drive/MyDrive/Thesis/POM-IMG/train_row4", exist_ok=True)
# os.makedirs("/content/drive/MyDrive/Thesis/POM-IMG/test_row3", exist_ok=True)
# os.makedirs("/content/drive/MyDrive/Thesis/POM-IMG/test_row4", exist_ok=True)

# # Clean Image_ID: remove only the row number (3 or 4), keep EW suffix
# def clean_image_id(image_id):
#     # Expecting format: PREFIX_IMGNUM_ROWNUM_VERSION_EW → remove ROWNUM
#     match = re.match(r"^([A-Z0-9]+)_([0-9]+)_(?:3|4)_(V\d+|VN)_([EW])$", image_id)
#     if match:
#         prefix, img_num, version, ew = match.groups()
#         return f"{prefix}_{img_num}_{version}_{ew}"
#     else:
#         return None  # Skip malformed IDs

# # Load CSVs
# train_row3 = pd.read_csv("/content/drive/MyDrive/Thesis/POM-IMG/train_row3.csv")
# train_row4 = pd.read_csv("/content/drive/MyDrive/Thesis/POM-IMG/train_row4.csv")
# test_row3 = pd.read_csv("/content/drive/MyDrive/Thesis/POM-IMG/test_row3.csv")
# test_row4 = pd.read_csv("/content/drive/MyDrive/Thesis/POM-IMG/test_row4.csv")

# # Helper function to save triplets
# def save_triplets(df, output_dir):
#     grouped = df.groupby("Image_ID")
#     count = 0
#     for image_id, group in grouped:
#         if set(group["Week"]) >= {"week-2", "week-3", "week-4"} and len(group) == 3:
#             cleaned_id = clean_image_id(image_id)
#             if cleaned_id:
#                 out_path = os.path.join(output_dir, f"{cleaned_id}.csv")
#                 group.to_csv(out_path, index=False)
#                 count += 1
#     print(f"Saved {count} triplets in {output_dir}")

# # Process all 4 splits
# save_triplets(train_row3, "/content/drive/MyDrive/Thesis/POM-IMG/train_row3")
# save_triplets(train_row4, "/content/drive/MyDrive/Thesis/POM-IMG/train_row4")
# save_triplets(test_row3, "/content/drive/MyDrive/Thesis/POM-IMG/test_row3")
# save_triplets(test_row4, "/content/drive/MyDrive/Thesis/POM-IMG/test_row4")


In [ ]:
# # fix names od cloumns (need it for later)

# def rename_columns(merged_df):
#     """Forcefully renames specific columns after merging to match the correct format."""
#     rename_dict = {
#         "Temp ֲ°C Avg.": "Temp °C Avg.",
#         "Humidity % Avg.": "Humidity % Avg.",
#         "Solar Radiation W/mֲ² Avg.": "Solar Radiation W/m² Avg.",
#         "Wind Speed m/sec Avg.": "Wind Speed m/sec Avg.",
#         "Wind Dir Avg.": "Wind Dir Avg.",
#         "TC ֲ°C Avg.": "TC °C Avg.",
#         "Atmospheric Pressure mb Avg.": "Atmospheric Pressure mb Avg.",
#         "Dew Point Cֲ° Avg.": "Dew Point C° Avg.",
#         "TC Cֲ° Min.": "TC C° Min.",
#         "Solar Radiation W/mֲ² Max.": "Solar Radiation W/m² Max.",
#         "Solar Radiation W/mֲ² Min.": "Solar Radiation W/m² Min."
#     }
#     return merged_df.rename(columns=rename_dict)

# # Folder containing your CSVs (change this to the correct folder)
# folder_path = "/content/drive/MyDrive/Thesis/POM-IMG/train_row3_SEF"

# # Loop through all folders and CSVs
# for subdir, _, files in os.walk(folder_path):
#     for file in files:
#         if file.endswith(".csv"):
#             csv_path = os.path.join(subdir, file)
#             try:
#                 df = pd.read_csv(csv_path)
#                 df = rename_columns(df)
#                 df.to_csv(csv_path, index=False, encoding='utf-8-sig')
#                 print(f"Renamed and saved: {csv_path}")
#             except Exception as e:
#                 print(f"Failed on {csv_path}: {e}")

In [ ]:
# #deleteing uneccary cloums

# # List of columns you want to **drop**
# columns_to_drop = ['Row_Number',	'Version',	'EW',	'Image_Count',	'Image_ID', 'Date',	'Time Aj', 'Date-D',	'Tree_ID', 'Tree_Prefix']

# # Path to the parent folder containing triplet CSVs (e.g., train_row3, test_row4, etc.)
# parent_folder = "/content/drive/MyDrive/Thesis/POM-IMG/test_row4"  # <- CHANGE THIS TO YOUR FOLDER

# # Loop through all subfolders and CSV files
# for subdir, _, files in os.walk(parent_folder):
#     for file in files:
#         if file.endswith(".csv"):
#             csv_path = os.path.join(subdir, file)
#             try:
#                 df = pd.read_csv(csv_path)
#                 df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
#                 df.to_csv(csv_path, index=False, encoding='utf-8-sig')
#                 print(f" Cleaned: {csv_path}")
#             except Exception as e:
#                 print(f" Failed on {csv_path}: {e}")

In [ ]:
# #Matching tripltes from each row 3 and 4

# # Paths to the row 3 and row 4 folders
# row3_folder = "/content/drive/MyDrive/Thesis/POM-IMG/train_row3"
# row4_folder = "/content/drive/MyDrive/Thesis/POM-IMG/train_row4"

# def get_triplet_ids(folder_path):
#     """Return a set of triplet IDs (filename without .csv) from a folder."""
#     return {filename.replace(".csv", "") for filename in os.listdir(folder_path) if filename.endswith(".csv")}

# def delete_unmatched(folder_path, valid_ids):
#     """Delete files in folder that are not in the set of valid triplet IDs."""
#     for filename in os.listdir(folder_path):
#         if filename.endswith(".csv"):
#             triplet_id = filename.replace(".csv", "")
#             if triplet_id not in valid_ids:
#                 os.remove(os.path.join(folder_path, filename))
#                 print(f" Deleted: {triplet_id} from {folder_path}")

# # Step 1: Get triplet IDs from both folders
# ids_row3 = get_triplet_ids(row3_folder)
# ids_row4 = get_triplet_ids(row4_folder)

# # Step 2: Keep only those triplets that are common in both
# matching_ids = ids_row3 & ids_row4  # Intersection

# # Step 3: Delete unmatched files
# delete_unmatched(row3_folder, matching_ids)
# delete_unmatched(row4_folder, matching_ids)

# print(f"\n Done! Matched Triplets: {len(matching_ids)}")

In [ ]:
# #seperate in each CSV to CSV for each week

# # Replace with your directory (e.g., test_row3, train_row4, etc.)
# input_dir = "/content/drive/MyDrive/Thesis/POM-IMG/train_row3"  # e.g., test_row3
# output_dir = "/content/drive/MyDrive/Thesis/POM-IMG/train_row3_S"

# # Make sure output exists
# os.makedirs(output_dir, exist_ok=True)

# for file in os.listdir(input_dir):
#     if file.endswith(".csv"):
#         triplet_path = os.path.join(input_dir, file)
#         triplet_df = pd.read_csv(triplet_path)

#         # Check that all weeks exist
#         if set(triplet_df["Week"]) >= {"week-2", "week-3", "week-4"}:
#             # Make subfolder
#             triplet_name = file.replace(".csv", "")
#             triplet_folder = os.path.join(output_dir, triplet_name)
#             os.makedirs(triplet_folder, exist_ok=True)

#             # Save each week separately
#             for week in ["week-2", "week-3", "week-4"]:
#                 week_df = triplet_df[triplet_df["Week"] == week]
#                 if not week_df.empty:
#                     week_df.to_csv(os.path.join(triplet_folder, f"{week}.csv"), index=False)

# print(" Done! Triplet folders with week-specific CSVs are created.")

In [ ]:
# #normlize columns

# #  Paths
# root_dir = "/content/drive/MyDrive/Thesis/POM-IMG/train_row3_S"  # Change to your actual path
# week_files = ["week-2.csv", "week-3.csv", "week-4.csv"]

# # Collect all metadata into one DataFrame (excluding non-numeric cols)
# all_metadata = []

# for folder in os.listdir(root_dir):
#     folder_path = os.path.join(root_dir, folder)
#     if not os.path.isdir(folder_path):
#         continue

#     for week_file in week_files:
#         week_path = os.path.join(folder_path, week_file)
#         if os.path.exists(week_path):
#             df = pd.read_csv(week_path)
#             # Drop non-numeric or identifier columns
#             meta = df.drop(columns=["Image_Path", "Image_ID", "Label", "Week"], errors="ignore")
#             all_metadata.append(meta)

# # Fit scaler on full metadata
# scaler = StandardScaler()
# full_meta_df = pd.concat(all_metadata, ignore_index=True)
# scaler.fit(full_meta_df)

# print(" Scaler fitted on all metadata.")

# # Normalize and overwrite each CSV
# for folder in os.listdir(root_dir):
#     folder_path = os.path.join(root_dir, folder)
#     if not os.path.isdir(folder_path):
#         continue

#     for week_file in week_files:
#         week_path = os.path.join(folder_path, week_file)
#         if os.path.exists(week_path):
#             df = pd.read_csv(week_path)
#             meta = df.drop(columns=["Image_Path", "Image_ID", "Label", "Week"], errors="ignore")
#             scaled_meta = scaler.transform(meta)

#             # Replace original columns
#             df.loc[:, meta.columns] = scaled_meta.astype('float32')

#             # Save back
#             df.to_csv(week_path, index=False)

# print(" All metadata normalized and saved.")

In [ ]:
#correlation Matrix
# df = pd.read_csv("/content/drive/MyDrive/Thesis/POM-IMG/train_row3.csv")

# # Drop non-numeric or irrelevant columns for correlation
# drop_cols = ["Week", "Image_Path", "Image_ID", "Label", "Row_Number","Image_Count"]
# df_numeric = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

# # Ensure all values are numeric (coerce errors)
# df_numeric = df_numeric.apply(pd.to_numeric, errors='coerce')

# # Drop columns with all NaNs (just in case)
# df_numeric = df_numeric.dropna(axis=1, how='all')

# # Compute correlation matrix
# correlation_matrix = df_numeric.corr()

# # Plot the heatmap
# plt.figure(figsize=(16, 14))
# sns.heatmap(correlation_matrix, annot=True, fmt=".2f",cmap="coolwarm", center=0, square=True, linewidths=.2)
# plt.title(" Feature Correlation Heatmap")
# plt.tight_layout()
# plt.show()

In [ ]:
# #create new columns and normelize them

# # Paths
# input_folder = "/content/drive/MyDrive/Thesis/POM-IMG/train_row3_S"  # original with base columns
# output_folder = "/content/drive/MyDrive/Thesis/POM-IMG/train_row3_SEF"
# os.makedirs(output_folder, exist_ok=True)

# # Base and engineered columns
# base_cols = [
#     'Image_Path', 'Label', 'Week',
#     'MeanTemperature', 'Temp °C Avg.', 'GradientMean', 'Humidity % Avg.',
#     'HaralickEnergy', 'Solar Radiation W/m² Avg.', 'ImageEntropy',
#     'Dew Point C° Avg.', 'Skewness', 'TC °C Avg.'
# ]
# engineered_cols = [
#     'Thermal_Deviation', 'Stress_Index', 'Energy_Ratio',
#     'Tree_Entropy_Surplus', 'Response_Index'
# ]
# all_keep_cols = base_cols + engineered_cols

# # Step 1: Engineer features and collect for normalization
# engineered_feature_list = []
# all_csv_paths = []

# def engineer(df):
#     df['Thermal_Deviation'] = df['MeanTemperature'] - df['Temp °C Avg.']
#     df['Stress_Index'] = df['GradientMean'] * df['Humidity % Avg.']
#     df['Energy_Ratio'] = df['HaralickEnergy'] / (df['Solar Radiation W/m² Avg.'] + 1e-5)
#     df['Tree_Entropy_Surplus'] = df['ImageEntropy'] - df['Dew Point C° Avg.']
#     df['Response_Index'] = (df['Skewness'] + df['ImageEntropy']) / (df['TC °C Avg.'] + 1e-5)
#     return df

# # Gather data for scaler
# for folder in os.listdir(input_folder):
#     folder_path = os.path.join(input_folder, folder)
#     if os.path.isdir(folder_path):
#         for file in os.listdir(folder_path):
#             if file.endswith(".csv"):
#                 file_path = os.path.join(folder_path, file)
#                 df = pd.read_csv(file_path)
#                 for col in base_cols:
#                     if col not in df.columns:
#                         df[col] = 0
#                 df = engineer(df)
#                 engineered_feature_list.append(df[engineered_cols])
#                 all_csv_paths.append((file_path, folder))  # save path for 2nd loop

# # Step 2: Fit scaler
# scaler = StandardScaler()
# engineered_full = pd.concat(engineered_feature_list)
# scaler.fit(engineered_full)

# # Step 3: Apply scaler & save
# for (file_path, folder) in all_csv_paths:
#     df = pd.read_csv(file_path)
#     for col in base_cols:
#         if col not in df.columns:
#             df[col] = 0
#     df = engineer(df)

#     df[engineered_cols] = scaler.transform(df[engineered_cols])

#     df = df[all_keep_cols]
#     output_subfolder = os.path.join(output_folder, folder)
#     os.makedirs(output_subfolder, exist_ok=True)
#     file_name = os.path.basename(file_path)
#     df.to_csv(os.path.join(output_subfolder, file_name), index=False)

# print("Engineered features recreated and normalized correctly.")


In [ ]:
# #turn lables 3 into 2

# def convert_labels_in_folder(root_dir):
#     for folder in os.listdir(root_dir):
#         folder_path = os.path.join(root_dir, folder)
#         if not os.path.isdir(folder_path):
#             continue

#         for week_file in ["week-2.csv", "week-3.csv", "week-4.csv"]:
#             csv_path = os.path.join(folder_path, week_file)
#             if not os.path.exists(csv_path):
#                 continue

#             df = pd.read_csv(csv_path)

#             # Convert label 3 → 2 (if column exists)
#             if "Label" in df.columns:
#                 df["Label"] = df["Label"].apply(lambda x: 2 if x == 3 else x)

#             # Save back
#             df.to_csv(csv_path, index=False, encoding="utf-8-sig")

#     print(f" Label conversion complete in: {root_dir}")

# convert_labels_in_folder("/content/drive/MyDrive/Thesis/POM-IMG/train_row3_SEF")
# convert_labels_in_folder("/content/drive/MyDrive/Thesis/POM-IMG/train_row4_SEF")
# convert_labels_in_folder("/content/drive/MyDrive/Thesis/POM-IMG/test_row3_SEF")
# convert_labels_in_folder("/content/drive/MyDrive/Thesis/POM-IMG/test_row4_SEF")


In [ ]:
#irragation cloum not used
# import os
# import pandas as pd

# def add_irrigation_to_csvs(root_folder):
#     for folder_name in os.listdir(root_folder):
#         folder_path = os.path.join(root_folder, folder_name)

#         if not os.path.isdir(folder_path):
#             continue

#         # Determine irrigation value from folder name
#         irrigation_value = 1 if folder_name.startswith("Y") else 0

#         for week_csv in ["week-2.csv", "week-3.csv", "week-4.csv"]:
#             csv_path = os.path.join(folder_path, week_csv)

#             if os.path.exists(csv_path):
#                 try:
#                     df = pd.read_csv(csv_path)
#                     df["Irrigation"] = irrigation_value
#                     df.to_csv(csv_path, index=False)
#                     print(f"Updated {csv_path} with Irrigation={irrigation_value}")
#                 except Exception as e:
#                     print(f" Failed to update {csv_path}: {e}")
#             else:
#                 print(f" Missing CSV: {csv_path}")

# add_irrigation_to_csvs("/content/drive/MyDrive/Thesis/POM-IMG/train_row4_SEF")

In [ ]:
class TripletDatasetByFolder(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform = transform
        self.sample_folders = [f for f in os.listdir(folder_path)
                               if os.path.isdir(os.path.join(folder_path, f))]

    def __len__(self):
        return len(self.sample_folders)

    def __getitem__(self, idx):
        folder_name = self.sample_folders[idx]
        folder_path = os.path.join(self.folder_path, folder_name)

        weeks = ["week-2.csv", "week-3.csv", "week-4.csv"]
        image_triplet = []
        meta_triplet = []
        label = None

        for i, week_file in enumerate(weeks):
            week_path = os.path.join(folder_path, week_file)
            if not os.path.exists(week_path):
                print(f"Missing file: {week_path}")
                image_triplet.append(torch.zeros(3, 224, 224))
                meta_triplet.append(torch.zeros(29))
                continue

            df = pd.read_csv(week_path)
            if df.empty or "Image_Path" not in df.columns:
                print(f"Empty or invalid file: {week_path}")
                image_triplet.append(torch.zeros(3, 224, 224))
                meta_triplet.append(torch.zeros(29))
                continue

            try:
                # Load image
                img_path = df.iloc[0]["Image_Path"]
                image = Image.open(img_path).convert("RGB")
                if self.transform:
                    image = self.transform(image)
                image_triplet.append(image)

                # Metadata + week
                meta = df.drop(columns=["Image_Path", "Image_ID", "Label"], errors="ignore")
                week_index = float(df["Week"].values[0][-1]) if "Week" in df.columns else (i + 2)
                meta["Week_Index"] = week_index
                meta_numeric = pd.to_numeric(meta.iloc[0], errors="coerce").fillna(0).values.astype("float32")
                meta_tensor = torch.tensor(meta_numeric[:29], dtype=torch.float32)
                meta_triplet.append(meta_tensor)

                if week_file == "week-4.csv" and "Label" in df.columns:
                    label = int(df.iloc[0]["Label"])

            except Exception as e:
                print(f"Error processing {week_path}: {e}")
                image_triplet.append(torch.zeros(3, 224, 224))
                meta_triplet.append(torch.zeros(17))

                            # Extract label from week 4
            if week_file == "week-4.csv":
                label = int(df.iloc[0]["Label"])
                if label == 3:
                    label = 2

        stacked_images = torch.stack(image_triplet)        # (3, C, H, W)
        stacked_metadata = torch.stack(meta_triplet)       # (3, 29)

        return stacked_images, stacked_metadata, torch.tensor(label, dtype=torch.long)

In [ ]:
#Data Loder row3 (north)
# Image transforms
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Directories
train_row3_dir = "/content/drive/MyDrive/Thesis/POM-IMG/train_row3_SEF"
test_row3_dir = "/content/drive/MyDrive/Thesis/POM-IMG/test_row3_SEF"

# Datasets
full_train_dataset3 = TripletDatasetByFolder(train_row3_dir, transform=image_transforms)
test_row3_dataset = TripletDatasetByFolder(test_row3_dir, transform=image_transforms)

# Split train into 80% train / 20% val
train_size = int(0.8 * len(full_train_dataset3))
val_size = len(full_train_dataset3) - train_size
train_row3_dataset, val_row3_dataset = random_split(full_train_dataset3, [train_size, val_size])

# Loaders
train_loader3 = DataLoader(train_row3_dataset, batch_size=16, shuffle=True)
val_loader3 = DataLoader(val_row3_dataset, batch_size=16, shuffle=True)
test_loader3 = DataLoader(test_row3_dataset, batch_size=16, shuffle=False)

print("Row 3 loaders ready!")
print(f"Train samples: {len(train_row3_dataset)} | Val samples: {len(val_row3_dataset)} | Test samples: {len(test_row3_dataset)}")
print(f"Train size: {len(train_row3_dataset)} | Batch size: {train_loader3.batch_size}")
print(f"Number of batches: {len(train_loader3)}")

In [ ]:
#Data Loder row4 (south)
# Image transforms
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Directories
train_row4_dir = "/content/drive/MyDrive/Thesis/POM-IMG/train_row4_SEF"
test_row4_dir = "/content/drive/MyDrive/Thesis/POM-IMG/test_row4_SEF"

# Datasets
full_train_dataset4 = TripletDatasetByFolder(train_row4_dir, transform=image_transforms)
test_row4_dataset = TripletDatasetByFolder(test_row4_dir, transform=image_transforms)

# Split train into 80% train / 20% val
train_size = int(0.8 * len(full_train_dataset4))
val_size = len(full_train_dataset4) - train_size
train_row4_dataset, val_row4_dataset = random_split(full_train_dataset4, [train_size, val_size])

# Loaders
train_loader4 = DataLoader(train_row4_dataset, batch_size=16, shuffle=True)
val_loader4 = DataLoader(val_row4_dataset, batch_size=16, shuffle=False)
test_loader4 = DataLoader(test_row4_dataset, batch_size=16, shuffle=False)

print("Row 3 loaders ready!")
print(f"Train samples: {len(train_row4_dataset)} | Val samples: {len(val_row4_dataset)} | Test samples: {len(test_row4_dataset)}")
print(f"Train size: {len(train_row4_dataset)} | Batch size: {train_loader4.batch_size}")
print(f"Number of batches: {len(train_loader4)}")

In [ ]:
# LSTM attention Block
class AttentionBlock(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out):
        # lstm_out: (B, T, H)
        attn_weights = self.attn(lstm_out)         # (B, T, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)  # Normalize across time
        context = (lstm_out * attn_weights).sum(dim=1)     # Weighted sum: (B, H)
        return context

In [ ]:
#CNN attention block
class TemporalImageAttention(nn.Module):
    def __init__(self, feature_dim, hidden_dim=128):
        super().__init__()
        self.attn_mlp = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        # x: (B, T, D) where T = 3 time steps (weeks), D = CNN feature dim (2048)
        scores = self.attn_mlp(x)                      # (B, T, 1)
        weights = torch.softmax(scores, dim=1)         # (B, T, 1)
        context = (x * weights).sum(dim=1)             # (B, D)
        return context

In [ ]:
# pomgrnate model with timm ConvNeXt

class PomRowModel(nn.Module):
    def __init__(self, num_metadata_features, hidden_dim=128, lstm_layers=3, num_classes=3):
        super().__init__()

        # ConvNeXt from timm
        self.cnn = timm.create_model('convnext_base', pretrained=True, num_classes=0)  # No FC layer
        cnn_output_dim = self.cnn.num_features  # Should be 1024 for convnext_base

        # LSTM for Metadata
        self.lstm = nn.LSTM(input_size=num_metadata_features,
                            hidden_size=hidden_dim,
                            num_layers=lstm_layers,
                            batch_first=True)

        # Dual Attention
        self.meta_attention = AttentionBlock(hidden_dim)
        self.img_attention = TemporalImageAttention(cnn_output_dim)

        # Fully Connected Head
        self.fc = nn.Sequential(
            nn.Linear(cnn_output_dim + hidden_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, imgs, meta):
        # imgs: (B, 3, C, H, W), meta: (B, 3, D)
        B, T, C, H, W = imgs.shape
        imgs = imgs.view(B * T, C, H, W)  # (B*T, C, H, W)
        cnn_feats = self.cnn(imgs)        # (B*T, F)
        cnn_feats = cnn_feats.view(B, T, -1)  # (B, 3, F)
        img_attn = self.img_attention(cnn_feats)  # (B, F)

        lstm_out, _ = self.lstm(meta)          # (B, 3, H)
        meta_attn = self.meta_attention(lstm_out)  # (B, H)

        combined = torch.cat([img_attn, meta_attn], dim=1)  # (B, F + H)
        return self.fc(combined)

In [ ]:
#traning loop
GPU = 0
SEED = 2023

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
os.environ['PYTHONHASHSEED'] = str(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")

class CombinedDynamicPenaltyLoss(nn.Module):
    def __init__(self, max_epoch=50, week_weights=None, base_weights=None):
        super().__init__()
        self.max_epoch = max_epoch
        self.current_epoch = 0
        self.penalty_matrix = None
        self.week_weights = week_weights or {2: 1.5, 3: 1.2, 4: 1.0}
        self.base_weights = base_weights

    def set_epoch(self, epoch):
        self.current_epoch = epoch

    def set_penalty_matrix(self, matrix):
        self.penalty_matrix = matrix

    def get_curriculum_scale(self):
        return min(0.1 + (self.current_epoch / self.max_epoch) * 0.9, 1.0)

    def forward(self, logits, targets, meta_weeks=None):
        assert self.penalty_matrix is not None, "Penalty matrix must be set each epoch"

        probs = F.softmax(logits, dim=1)
        log_probs = F.log_softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=logits.size(1)).float()

        ce_loss = -torch.sum(log_probs * targets_one_hot, dim=1)
        if self.base_weights is not None:
            ce_loss = ce_loss * self.base_weights[targets]

        preds = logits.argmax(dim=1)
        pred_conf = probs.gather(1, preds.unsqueeze(1)).squeeze(1)
        is_wrong = (preds != targets).float()

        curriculum_scale = self.get_curriculum_scale()
        scaled_penalty_matrix = self.penalty_matrix * curriculum_scale
        penalties = scaled_penalty_matrix[targets, preds] * pred_conf * is_wrong

        if meta_weeks is not None:
            week_factors = torch.tensor(
                [self.week_weights.get(int(w), 1.0) for w in meta_weeks],
                device=logits.device
            )
            penalties = penalties * week_factors

        return (ce_loss + penalties).mean()

def generate_dynamic_penalty_matrix(epoch, num_classes, device):
    # Example: increase penalty for class 0→2 and 2→0 as training progresses
    base = torch.zeros(num_classes, num_classes)
    base[0, 2] = 1.5
    base[2, 0] = 1.5
    scale = min(epoch / 10.0, 1.0)
    return base * scale

def compute_class_weights(dataloader, num_classes, device):
    all_labels = []
    for _, _, labels in dataloader:
        all_labels.extend(labels.tolist())

    label_counts = Counter(all_labels)
    total = sum(label_counts.values())
    weights = []

    for i in range(num_classes):
        class_count = label_counts.get(i, 1)  # Avoid division by zero
        weight = total / (num_classes * class_count)
        weights.append(weight)

    # Normalize weights (optional)
    weights = np.array(weights)
    weights = weights / np.mean(weights)

    return torch.tensor(weights, dtype=torch.float32).to(device)


def validate(model, val_loader, criterion, device):
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, metadata, labels in val_loader:
            images = images.to(device)
            metadata = metadata.to(device)
            labels = labels.to(device)

            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    return val_loss / len(val_loader)


def train_model(model, dataloader, val_loader, num_epochs=10, lr=0.0003, num_classes=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    class_weights = compute_class_weights(dataloader, num_classes, device)
    print(f" Normalized Class Weights: {class_weights.tolist()}")

    criterion = CombinedDynamicPenaltyLoss(max_epoch=num_epochs, base_weights=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

    best_val_loss = float('inf')
    best_model_weights = None

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        print(f"\n Epoch {epoch+1}/{num_epochs}")

        criterion.set_epoch(epoch)
        dynamic_penalty_matrix = generate_dynamic_penalty_matrix(epoch, num_classes, device)
        criterion.set_penalty_matrix(dynamic_penalty_matrix.to(device))

        for i, (images, metadata, labels) in enumerate(dataloader):
            images = images.to(device)
            metadata = metadata.to(device)
            labels = labels.to(device)

            initial_weights = model.cnn.stem[0].weight.clone().detach()

            optimizer.zero_grad()
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            updated_weights = model.cnn.stem[0].weight

            total_loss += loss.item()

        avg_train_loss = total_loss / len(dataloader)
        print(f" Epoch {epoch+1} Avg Train Loss: {avg_train_loss:.4f}")

        val_loss = validate(model, val_loader, criterion, device)
        print(f" Validation Loss: {val_loss:.4f}")
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_weights = model.state_dict()
            print(f" New best model saved (val_loss = {val_loss:.4f})")

    # Load best weights before returning
    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print(" Loaded best model weights from memory!")

    print(" Training complete!")
    return model


In [ ]:
#run model
# Set number of metadata features
num_metadata_features = 18  # Set based on your dataset

# Initialize model
model = PomRowModel(num_metadata_features=num_metadata_features)

# Train the model
train_model(model, train_loader3, val_loader3, num_epochs=30, lr=0.001)


In [ ]:
#test model
def test_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            image_seq, metadata_seq, labels = batch

            image_seq = image_seq.to(device)       # shape: (B, 3, C, H, W)
            metadata_seq = metadata_seq.to(device) # shape: (B, 3, D)
            labels = labels.to(device)             # shape: (B,)

            outputs = model(image_seq, metadata_seq)  # shape: (B, num_classes)
            _, preds = torch.max(outputs, 1)       # Get predicted class indices

            # Extend lists directly with tensors, converting to NumPy later
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    # Convert to numpy arrays after the loop
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Accuracy
    accuracy = 100 * np.sum(all_preds == all_labels) / len(all_labels)
    print(f" Test Accuracy: {accuracy:.2f}%")

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    print("\n Confusion Matrix:")
    print(cm)

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()

    # Classification report
    print("\n Classification Report:")
    print(classification_report(all_labels, all_preds, digits=3))

    return all_preds, all_labels

In [ ]:
#Run test model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# test_loader should yield: (image_seq, metadata_seq, labels)
preds, labels = test_model(model, test_loader3, device)

In [ ]:
# Save the trained model
model_save_path = "/content/drive/MyDrive/Thesis/POM-IMG/row4_convnext.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

In [ ]:
#compaer sides of the field AA metric
# Load models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

row3_model = CitrusRowModel(num_metadata_features=17).to(device)
row4_model = CitrusRowModel(num_metadata_features=17).to(device)

row3_model.load_state_dict(torch.load("/content/drive/MyDrive/Thesis/POM-IMG/row3and4_convnext_atten_16batch_30epochs/row3_convnext.pth", map_location=device))
row4_model.load_state_dict(torch.load("/content/drive/MyDrive/Thesis/POM-IMG/row3and4_convnext_atten_16batch_30epochs/row4_convnext.pth", map_location=device))

row3_model.eval()
row4_model.eval()

#  Set test folder path and load folder names
test_row3_dir = "/content/drive/MyDrive/Thesis/POM-IMG/test_row3_SEF"
folder_names = sorted([
    f for f in os.listdir(test_row3_dir)
    if os.path.isdir(os.path.join(test_row3_dir, f))
])

#  Run predictions and compare
agree_and_correct = 0
total_samples = 0
disagreements = []
all_preds = []

for batch_row3, batch_row4 in tqdm(zip(test_loader3, test_loader4), total=len(test_loader3)):
    images_r3, meta_r3, labels_r3 = batch_row3
    images_r4, meta_r4, labels_r4 = batch_row4

    images_r3 = images_r3.to(device)
    meta_r3 = meta_r3.to(device)
    images_r4 = images_r4.to(device)
    meta_r4 = meta_r4.to(device)
    labels = labels_r3.to(device)  # Assuming same labels for both

    with torch.no_grad():
        preds_r3 = row3_model(images_r3, meta_r3).argmax(dim=1)
        preds_r4 = row4_model(images_r4, meta_r4).argmax(dim=1)

    min_batch_size = min(len(preds_r3), len(preds_r4))

    for i in range(min_batch_size):
        pred3 = preds_r3[i].item()
        pred4 = preds_r4[i].item()
        label = labels[i].item()
        folder = folder_names[total_samples]  # Match folder name to sample

        if pred3 == pred4 and pred3 == label:
            agree_and_correct += 1
        elif pred3 != pred4:
            disagreements.append((folder, pred3, pred4, label))

        all_preds.append((folder, pred3, pred4, label))
        total_samples += 1

# Report results
accuracy = agree_and_correct / total_samples
print(f"\n✅ Agreement Accuracy: {accuracy:.4f}")
print(f"❌ Disagreements: {len(disagreements)} / {total_samples}")

# Save to CSV
results_df = pd.DataFrame(all_preds, columns=["Folder", "Row3_Pred", "Row4_Pred", "True_Label"])
results_df.to_csv("/content/drive/MyDrive/Thesis/POM-IMG/model_comparison_results.csv", index=False)
print(" Saved: model_comparison_results.csv")